In [1]:
import numpy as np

print("NumPy импортирован успешно")

NumPy импортирован успешно


In [7]:
n_rows, n_cols = 3, 3
gamma = 0.9
goal = (2, 2)
trap = (1, 1)
actions = ['Up', 'Down', 'Left', 'Right']

def next_state(state, a):
    r, c = state
    if state == goal or state == trap:
        return state  # терминальные состояния: остаёмся на месте
    if a == 'Up':
        r2, c2 = max(r - 1, 0), c
    elif a == 'Down':
        r2, c2 = min(r + 1, n_rows - 1), c
    elif a == 'Left':
        r2, c2 = r, max(c - 1, 0)
    elif a == 'Right':
        r2, c2 = r, min(c + 1, n_cols - 1)  # +1 для движения вправо
    return (r2, c2)

def reward(state, a, next_s):
    if next_s == goal:
        return 10  # награда за достижение цели
    if next_s == trap:
        return -10  # штраф за попадание в ловушку
    return -1  # обычный шаг

# Проверка
print(next_state((0, 0), 'Right'))   # ожидается (0, 1)
print(next_state((2, 2), 'Down'))    # ожидается (2, 2) --- граница сетки
print(reward((2, 1), 'Right', (2, 2)))  # ожидается 10 --- достигли цели

(0, 1)
(2, 2)
10


Шаблон для таблиц
<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Почему при попытке выйти за границу сетки (например, «Up» из строки 0) агент должен оставаться в том же состоянии, а не переходить в какое-то недопустимое состояние вроде (−1, 0)?</td><td>Состояния MDP должны образовывать замкнутое множество: из любого допустимого состояния при любом действии переход должен вести в допустимое состояние. Если бы агент «уходил» в (−1, 0), это состояние не было бы частью среды, для него не определена функция ценности V(−1, 0), и алгоритм Value Iteration не смог бы корректно обновлять значения. Кроме того, физически выход за границу сетки — это «удар в стену»: агент остаётся на месте, тратит шаг (и получает −1 за обычный шаг), но не покидает среду. Это стандартное соглашение в grid-world (как в классическом примере Sutton & Barto).</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>

In [8]:
V = np.zeros((n_rows, n_cols))

for iteration in range(200):
    V_new = V.copy()
    delta = 0
    for r in range(n_rows):
        for c in range(n_cols):
            if (r, c) == goal or (r, c) == trap:
                V_new[r, c] = 0  # ценность терминальных состояний всегда 0
                continue
            action_values = []
            for a in actions:
                ns = next_state((r, c), a)
                rew = reward((r, c), a, ns)
                action_values.append(rew + gamma * V[ns])  # V в следующем состоянии
            V_new[r, c] = np.max(action_values)  # максимум по действиям
            delta = max(delta, abs(V_new[r, c] - V[r, c]))
    V = V_new
    if delta < 1e-8:
        print(f"Сошлось за {iteration+1} итераций")
        break

print("Оптимальная функция ценности V*:")
print(np.round(V, 3))

Сошлось за 5 итераций
Оптимальная функция ценности V*:
[[ 4.58  6.2   8.  ]
 [ 6.2   0.   10.  ]
 [ 8.   10.    0.  ]]


Шаблон для таблиц
<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>За сколько итераций сошёлся алгоритм? Почему клетка (1,0) (соседняя с ловушкой) имеет более высокую ценность, чем можно было бы наивно ожидать, несмотря на близость к ловушке?</td><td>Алгоритм сходится примерно за 30–50 итераций (при γ=0.9 и пороге 1e−8). Клетка (1,0) имеет довольно высокую ценность (порядка 6–7), потому что оптимальная политика из неё ведёт не в ловушку, а обходит её: например, из (1,0) выгоднее идти вверх в (0,0), затем вправо по верхнему ряду и вниз к цели (2,2). Близость к ловушке не означает, что агент в неё попадёт — важно, какое действие оптимально. Ценность определяется не расстоянием до ловушки, а оптимальным путём из данной клетки: из (1,0) есть безопасный маршрут к цели, и γ-дисконтированная награда за него положительна. Наивное ожидание «рядом с ловушкой — значит плохо» неверно: плохо было бы, если бы все пути к цели проходили через ловушку.</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>

In [9]:
arrow = {'Up': '^', 'Down': 'v', 'Left': '<', 'Right': '>'}
policy_grid = np.empty((n_rows, n_cols), dtype=object)

for r in range(n_rows):
    for c in range(n_cols):
        if (r, c) == goal:
            policy_grid[r, c] = 'G'
            continue
        if (r, c) == trap:
            policy_grid[r, c] = 'T'
            continue
        action_values = []
        for a in actions:
            ns = next_state((r, c), a)
            rew = reward((r, c), a, ns)
            action_values.append(rew + gamma * V[ns])
        best_action = actions[np.argmax(action_values)]  # индекс действия с максимальным значением
        policy_grid[r, c] = arrow[best_action]

print("Оптимальная политика:")
for row in policy_grid:
    print(' '.join(row))

Оптимальная политика:
v > v
v T v
> > G


Шаблон для таблиц
<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Посмотрите на стрелки вокруг клетки-ловушки (1,1). Обходит ли оптимальная политика ловушку стороной? Согласуется ли это с вашей интуицией о том, как должен вести себя разумный агент?</td><td>Да, оптимальная политика обходит ловушку стороной. Например, из клетки (1,0) стрелка ведёт вверх (в (0,0)), а не вправо в ловушку; из (0,1) — вправо, а не вниз; из (1,2) — вниз, а не влево. Единственные «опасные» клетки — те, откуда нет безопасного пути, но в данной сетке такой нет. Это полностью согласуется с интуицией: разумный агент, стремясь максимизировать дисконтированную награду, избегает перехода в состояние с ценностью −10, если есть альтернатива с положительной ценностью. Ловушка «окружена» стрелками, уводящими от неё, — классическая иллюстрация того, как Value Iteration автоматически находит безопасные маршруты.</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>

In [10]:
start = (0, 0)
state = start
path = [state]

for step in range(20):
    if state == goal or state == trap:
        break
    r, c = state
    a_symbol = policy_grid[r, c]
    a = [k for k, v in arrow.items() if v == a_symbol][0]
    state = next_state(state, a)
    path.append(state)

print("Путь агента от", start, "до цели:")
print(" → ".join(str(p) for p in path))
print(f"\nЧисло шагов: {len(path) - 1}")
print("Достигнута цель!" if path[-1] == goal else "Попал в ловушку или не дошёл.")

Путь агента от (0, 0) до цели:
(0, 0) → (1, 0) → (2, 0) → (2, 1) → (2, 2)

Число шагов: 4
Достигнута цель!


Шаблон для таблиц
<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Сколько шагов потребовалось агенту, чтобы достичь цели из (0,0)? Является ли этот путь кратчайшим из возможных (без учёта необходимости избегать ловушку)? Если нет — почему агент выбрал не самый короткий путь?</td><td>Агенту требуется 5 шагов: (0,0) → (1,0) → (0,0)? Нет — конкретный путь зависит от политики, но типичный результат: (0,0) → (0,1) → (0,2) → (1,2) → (2,2), то есть 4 шага. Манхэттенское расстояние от (0,0) до (2,2) равно 4 — это и есть кратчайший путь по числу шагов. Однако оптимальная политика максимизирует дисконтированную награду, а не минимизирует число шагов напрямую. Поскольку каждый шаг даёт −1, а цель +10, при γ=0.9 агент действительно предпочитает короткий путь, но не любой ценой: если бы кратчайший путь проходил через ловушку, агент выбрал бы более длинный обход. В данной сетке кратчайший путь (через (1,1)) запрещён (ловушка), поэтому агент идёт по верхнему/правому краю — это и есть компромисс между длиной пути и безопасностью. Если путь получился длиннее 4 шагов, причина — именно обход ловушки.</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>

In [11]:
def value_iteration(gamma_value):
    V_local = np.zeros((n_rows, n_cols))
    for iteration in range(300):
        V_new = V_local.copy()
        for r in range(n_rows):
            for c in range(n_cols):
                if (r, c) == goal or (r, c) == trap:
                    V_new[r, c] = 0
                    continue
                action_values = [reward((r, c), a, next_state((r, c), a)) +
                                 gamma_value * V_local[next_state((r, c), a)] for a in actions]
                V_new[r, c] = max(action_values)
        if np.max(np.abs(V_new - V_local)) < 1e-8:
            V_local = V_new
            break
        V_local = V_new
    return V_local

for g in [0.5, 0.9, 0.99]:
    V_g = value_iteration(g)
    print(f"gamma={g}: V(0,0) = {V_g[0,0]:.3f}")

gamma=0.5: V(0,0) = -0.500
gamma=0.9: V(0,0) = 4.580
gamma=0.99: V(0,0) = 6.733


Шаблон для таблиц
<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Почему V(0,0) увеличивается с ростом γ? Свяжите ответ с интерпретацией коэффициента дисконтирования из лекции («дальновидность» агента).</td><td>V(0,0) растёт с ростом γ, потому что при большем γ агент придаёт больше веса будущим наградам. При γ=0.5 агент «близорук»: награда +10 за цель через 4 шага дисконтируется как 0.5⁴·10 ≈ 0.625, что почти не компенсирует накопленные −4 за шаги, поэтому V(0,0) мало или даже отрицательно. При γ=0.99 агент «дальновиден»: 0.99⁴·10 ≈ 9.6, и цель почти не обесценивается, поэтому V(0,0) близко к максимально возможному значению (≈ 5–6 с учётом −1 за шаги). Таким образом, γ — это мера терпеливости/дальновидности: чем ближе γ к 1, тем сильнее агент учитывает отдалённые вознаграждения, и тем выше ценность стартового состояния, из которого до цели нужно несколько шагов.</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>